# Commodities Prices (TTF, EUA, API2)
Retrieve daily prices for 2022-2025 and preview for cleaning/merging.
Tickers (Yahoo Finance):
- gas_price_ttf: TTF=F
- co2_price_eua: CFI=F
- coal_price_api2: MTF=F


In [2]:
import yfinance as yf
import pandas as pd
from pathlib import Path

TICKERS = {
    'gas_price_ttf': 'TTF=F',
    'co2_price_eua': 'CFI=F',
    'coal_price_api2': 'MTF=F',
}
START = '2022-01-01'
END = '2025-12-31'
DATA_DIR = Path('..') / 'data'


ModuleNotFoundError: No module named 'yfinance'

In [ ]:
def fetch_prices(tickers, start=START, end=END, interval='1d'):
    frames = []
    for col, ticker in tickers.items():
        df = yf.download(ticker, start=start, end=end, interval=interval, auto_adjust=False, progress=False)
        if df.empty:
            print(f'No data for {ticker}')
            continue
        df = df.reset_index().rename(columns={'Date': 'timestamp', 'Adj Close': col})
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
        frames.append(df[['timestamp', col]])
    if not frames:
        return pd.DataFrame()
    merged = frames[0]
    for df in frames[1:]:
        merged = merged.merge(df, on='timestamp', how='outer')
    merged = merged.sort_values('timestamp').drop_duplicates(subset=['timestamp'], keep='last')
    return merged

commodities = fetch_prices(TICKERS)
commodities.head()


In [ ]:
commodities.describe(include='all')


In [ ]:
# Save to parquet for merging
out_path = DATA_DIR / 'commodities.parquet'
out_path.parent.mkdir(parents=True, exist_ok=True)
commodities.to_parquet(out_path, compression='zstd', index=False)
out_path


## Notes / Cleaning hints
- Prices are daily; missing days (weekends/holidays) are expected.
- If occasional nulls appear, forward-fill is reasonable for merging to hourly data.
- Outliers: commodity prices can spike; clip only if they break downstream models.
- Merge compatibility: saved parquet uses `timestamp` with UTC to align with other datasets.


## Execution status
(Run this notebook with internet access to pull Yahoo Finance data; not executed in this environment.)
